# Обучение и инференс тела голов модели DLT (Ромашка v3.0)

## Подготовка данных

Полный цикл подготовки из уже скачанных данных в DLT:

- [examples/load_data/load_test_for_dlt_head.ipynb](examples/load_data/load_test_for_dlt_head.ipynb)
- [examples/load_data/load_train_for_dlt_head.ipynb](examples/load_data/load_train_for_dlt_head.ipynb)

## Импорты и проверка работоспособности кластера

In [ ]:
import os
import sys

sys.path.insert(0, os.path.abspath("/opt/clients"))

import osiris

In [ ]:
osiris.list()

In [ ]:
for job in osiris.list()["jobs"]:
    osiris.delete(job["job"])

## Запуск обучения модели

In [ ]:
job = osiris.create(
    name="accelerate-fmlib-4x4-dlt",
    image="registry.ca.sbrf.ru/ci02684173/ci02697916/notebooks/python3.12/cuda12.4/d-03.000.00:d-03.000.00-gigachat",
    restart=False,
    pool="public",
    command=[
        "/home/datalab/nfs/amazmefmlib_hgx_venv/bin/accelerate",
        "launch",
    ],
    args=[
        "--mixed_precision",
        "no",
        "--dynamo_backend",
        "no",
        "--num_machines",
        "4",
        "--num_processes",
        "16",
        "--main_process_ip",
        "$(MASTER_ADDR).datalab.svc.cluster.local",
        "--main_process_port",
        "$(MASTER_PORT)",
        "--machine_rank",
        "$(RANK)",
        "/home/datalab/nfs/amazmefmlib/fmlib/training/train.py",
        "--config-dir=/home/datalab/nfs/amazmefmlib/examples/configs/train",
        "--config-name=dlt_head",
        "+exact_weights_file=/home/datalab/nfs/new-camomile-artifacts/DLT/ds_weights/weights.bin",
    ],
    envs={
        "PYTHONPATH": "/home/datalab/nfs/amazmefmlib_hgx_venv",
        "OMP_NUM_THREADS": "24",
        "NCCL_DEBUG": "INFO",
    },
    num_nodes=4,
    num_gpus=4,
    type="pytorchjob",
)
job

In [ ]:
osiris.state(job["job"])

In [ ]:
osiris.logs(job["job"], tail_lines=32, is_master=True)